In [1]:
from google.colab import drive

drive.mount('/content/drive/')
%cd /content/drive/MyDrive/Dacon/물가 예측 2차

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/MyDrive/Dacon/물가 예측 2차


In [1]:
import pandas as pd

# Load the data
df1 = pd.read_csv('C:/Users/dpwl1/Downloads/data/data/train/train_1.csv')
df2 = pd.read_csv('C:/Users/dpwl1/Downloads/data/data/train/train_2.csv')

# Define the date parsing function
def parse_date(date):
    year = int(date[:4])
    month = int(date[4:6])
    period = date[6:]

    if period == '하순':
        day = 5
    elif period == '중순':
        day = 15
    elif period == '상순':
        day = 25
    else:
        day = 1

    return pd.Timestamp(year=year, month=month, day=day)

# Apply date parsing to both datasets
df1['datetime'] = df1['YYYYMMSOON'].apply(parse_date)
df2['datetime'] = df2['YYYYMMSOON'].apply(parse_date)

# Create separate DataFrames for each unique item
# Use '품목(품종)명' for df1 and '품목명' for df2
item_dfs1 = {item: df1[df1['품목(품종)명'] == item][['datetime', '평균가격(원)']].copy() for item in df1['품목(품종)명'].unique()}
item_dfs2 = {item: df2[df2['품목명'] == item][['datetime', '평균가격(원)']].copy() for item in df2['품목명'].unique()}

# Merge both item_dfs dictionaries based on unique item names
merged_item_dfs = {}

for item in set(item_dfs1.keys()).union(item_dfs2.keys()):
    # Get data for the current item from both df1 and df2, or an empty DataFrame if it doesn't exist in one of them
    item_df1 = item_dfs1.get(item, pd.DataFrame(columns=['datetime', '평균가격(원)']))
    item_df2 = item_dfs2.get(item, pd.DataFrame(columns=['datetime', '평균가격(원)']))

    # Concatenate data for the item and sort by datetime
    merged_item_dfs[item] = pd.concat([item_df1, item_df2]).sort_values(by='datetime').reset_index(drop=True)

C:\Users\dpwl1\AppData\Local\Temp\ipykernel_24060\2532572965.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_item_dfs[item] = pd.concat([item_df1, item_df2]).sort_values(by='datetime').reset_index(drop=True)
C:\Users\dpwl1\AppData\Local\Temp\ipykernel_24060\2532572965.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_item_dfs[item] = pd.concat([item_df1, item_df2]).sort_values(by='datetime').reset_index(drop=True)
C:\Users\dpwl1\AppData\Local\Temp\ipykernel_24060\2532572965

In [2]:
merged_item_dfs

{'배':       datetime  평균가격(원)
 0   2018-01-05  28324.0
 1   2018-01-15  28290.0
 2   2018-01-25  28312.0
 3   2018-02-05  28129.0
 4   2018-02-15  28294.0
 ..         ...      ...
 175 2022-11-15  25345.0
 176 2022-11-25  25248.0
 177 2022-12-05  25973.0
 178 2022-12-15  26504.0
 179 2022-12-25  25870.0
 
 [180 rows x 2 columns],
 '양파':       datetime      평균가격(원)
 0   2018-01-05  1036.444444
 1   2018-01-15  1041.222222
 2   2018-01-25  1144.428571
 3   2018-02-05   922.142857
 4   2018-02-15  1098.166667
 ..         ...          ...
 175 2022-11-15  1440.750000
 176 2022-11-25  1485.444444
 177 2022-12-05  1423.900000
 178 2022-12-15  1399.250000
 179 2022-12-25  1438.111111
 
 [180 rows x 2 columns],
 '감자 수미':       datetime       평균가격(원)
 0   2018-01-05  50243.000000
 1   2018-01-15  48283.777778
 2   2018-01-25  44170.285714
 3   2018-02-05  60486.714286
 4   2018-02-15  59133.000000
 ..         ...           ...
 175 2022-11-15  41064.000000
 176 2022-11-25  40469.777778
 177 202

In [3]:
from sklearn.model_selection import train_test_split

# Split each item DataFrame into train and test
train_test_data = {}
for item, data in merged_item_dfs.items():
    train, test = train_test_split(data, test_size=0.2, shuffle=False)
    train_test_data[item] = {'train': train, 'test': test}

In [5]:
import statsmodels.api as sm
import itertools
import warnings
warnings.filterwarnings('ignore')

def optimize_sarima(y, p_values, d_values, q_values, P_values, D_values, Q_values, s_values):
    best_aic = float('inf')
    best_params = None
    best_model = None

    # Grid search over the provided ranges for parameters
    for order in itertools.product(p_values, d_values, q_values):
        for seasonal_order in itertools.product(P_values, D_values, Q_values, s_values):
            try:
                model = sm.tsa.statespace.SARIMAX(
                    y,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(disp=False)

                # Check if the model has a lower AIC
                if model.aic < best_aic:
                    best_aic = model.aic
                    best_params = (order, seasonal_order)
                    best_model = model
            except:
                continue

    return best_model, best_params

# Parameter ranges
p_values = range(0, 2)       # AR order
d_values = range(0, 2)       # Differencing order
q_values = range(0, 2)       # MA order
P_values = range(0, 2)       # Seasonal AR order
D_values = range(0, 2)       # Seasonal differencing order
Q_values = range(0, 2)       # Seasonal MA order
s_values = [3, 9, 36]           # Seasonal period (quarterly or yearly)

models = {}
params = {}

for item, data in train_test_data.items():
    # y_train의 인덱스를 datetime으로 설정
    y_train = data['train'].set_index('datetime')['평균가격(원)']

    # Find the optimal model and parameters
    best_model, best_params = optimize_sarima(
        y_train, p_values, d_values, q_values, P_values, D_values, Q_values, s_values
    )

    models[item] = best_model
    params[item] = best_params
    print(f"Best model for {item}: order={best_params[0]}, seasonal_order={best_params[1]}, AIC={best_model.aic}")



Best model for 배: order=(0, 1, 1), seasonal_order=(1, 1, 1, 36), AIC=1281.4231799310414
Best model for 양파: order=(0, 1, 1), seasonal_order=(0, 1, 1, 36), AIC=936.4397707254037
Best model for 감자 수미: order=(0, 1, 1), seasonal_order=(0, 1, 1, 36), AIC=1367.244417698311
Best model for 상추: order=(1, 1, 1), seasonal_order=(0, 1, 1, 36), AIC=939.5584759013798
Best model for 대파(일반): order=(1, 1, 1), seasonal_order=(1, 0, 1, 36), AIC=867.0804561445177
Best model for 무: order=(1, 1, 1), seasonal_order=(1, 1, 1, 36), AIC=1325.906862240014
Best model for 배추: order=(1, 1, 1), seasonal_order=(0, 1, 1, 36), AIC=1311.2973417777125
Best model for 깐마늘(국산): order=(0, 1, 1), seasonal_order=(0, 1, 1, 36), AIC=1491.2139926189832
Best model for 사과: order=(1, 1, 1), seasonal_order=(1, 0, 1, 36), AIC=969.6757735369694
Best model for 건고추: order=(0, 1, 1), seasonal_order=(1, 1, 1, 36), AIC=1733.8974732353965


In [6]:
# Forecasting on test set
predictions = {}
for item, model in models.items():
    test_data = train_test_data[item]['test']
    forecast = model.forecast(steps=len(test_data))
    predictions[item] = forecast

In [7]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# NMAE 계산 함수 정의
def nmae_not_dict(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    nmae = mae / np.mean(np.abs(y_true))  # 정규화된 MAE 계산
    return nmae

# 테스트 데이터셋에 대한 NMAE 성능 평가
def evaluate_predictions(predictions, train_test_data):
    total_score = 0
    item_count = len(predictions)

    for item, forecast in predictions.items():
        # 실제 테스트 데이터셋 값 불러오기
        y_true = train_test_data[item]['test']['평균가격(원)'].values

        # 예측값과 실제값의 NMAE 계산
        nmae_score = nmae_not_dict(y_true, forecast)
        print(f"{item}: NMAE = {nmae_score}")

        # 총 NMAE 계산을 위해 합산
        total_score += nmae_score

    # 전체 평균 NMAE 출력
    average_nmae = total_score / item_count
    print(f"Average NMAE across all items: {average_nmae}")

# NMAE 성능 평가 함수 호출
evaluate_predictions(predictions, train_test_data)

배: NMAE = 0.10917931844133522
양파: NMAE = 0.8152474566178884
감자 수미: NMAE = 0.1064223838593919
상추: NMAE = 0.1551286007313419
대파(일반): NMAE = 3.052115681197406e+82
무: NMAE = 0.4163568631628026
배추: NMAE = 0.27199609133663477
깐마늘(국산): NMAE = 0.139419169069269
사과: NMAE = 3.6552751419773526e+91
건고추: NMAE = 0.3526465551900584
Average NMAE across all items: 3.6552751450294685e+90
